# Preprocesamiento  y Persistencia de Modelos
## Dataset: Datos sintéticos — Cancelación de Suscripción (Streaming)

Ttécnicas:

1. Generación de datos sintéticos
2. Estrategias de imputación: media, mediana, moda y constante
3. Limpieza por reglas de dominio: rangos válidos, valores imposibles
4. Codificación One-Hot (`pd.get_dummies`) como alternativa a LabelEncoder
5. Entrenamiento básico del modelo
6. **Persistencia del modelo** con `joblib` (guardar y cargar)
7. Predicción con el modelo cargado desde disco



In [1]:
%pip install numpy pandas matplotlib scikit-learn joblib


In [2]:
# Importaciones para preprocesamiento, modelado y persistencia.
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (accuracy_score, precision_score,
                                     recall_score, f1_score)

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42

---
## 1. Generación del Dataset Sintético

El problema es predecir si un suscriptor de una plataforma de streaming
**cancelará su plan** (`cancela = 1`) o **continuará** (`cancela = 0`).
Este es un caso clásico de clasificación binaria conocido como *churn prediction*.

In [3]:
# Generación de perfiles de usuario con distribuciones aproximadas a datos reales.
rng = np.random.default_rng(seed=SEED)
N   = 1_500

df_raw = pd.DataFrame({
    "meses_suscrito"        : rng.integers(1, 60, size=N).astype(float),
    "peliculas_vistas_mes"  : rng.integers(0, 30, size=N).astype(float),
    "series_vistas_mes"     : rng.integers(0, 15, size=N).astype(float),
    "horas_uso_diario"      : rng.normal(loc=2.5, scale=1.5, size=N).round(1),
    "calificacion_promedio" : rng.normal(loc=3.5, scale=1.0, size=N).round(1),
    "dispositivos_vinculados": rng.integers(1, 5, size=N).astype(float),
    "interrupciones_mes"    : rng.integers(0, 20, size=N).astype(float),
    "quejas_soporte"        : rng.integers(0, 5,  size=N).astype(float),
    "precio_plan_mensual"   : rng.normal(loc=120, scale=40, size=N).round(2),
    # Variable categórica: tipo de plan contratado
    "tipo_plan"             : rng.choice(["Basico", "Estandar", "Premium"], size=N),
})

# Regla de negocio + ruido: usuarios poco activos y con quejas tienden a cancelar.
score_cancelacion = (
    (df_raw["horas_uso_diario"]    < 1.0).astype(int) * 2 +
    (df_raw["quejas_soporte"]      >= 3).astype(int)  * 2 +
    (df_raw["calificacion_promedio"]< 2.5).astype(int)    +
    (df_raw["peliculas_vistas_mes"] < 3).astype(int)
)
ruido = rng.integers(0, 2, size=N)
df_raw["cancela"] = ((score_cancelacion + ruido) >= 3).astype(int)

print(f"Dataset generado: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas")
print(f"Distribución objetivo:\n{df_raw['cancela'].value_counts()}")
df_raw.head()

Dataset generado: 1500 filas × 11 columnas
Distribución objetivo:
cancela
0    1026
1     474
Name: count, dtype: int64


,meses_suscrito,peliculas_vistas_mes,series_vistas_mes,horas_uso_diario,calificacion_promedio,dispositivos_vinculados,interrupciones_mes,quejas_soporte,precio_plan_mensual,tipo_plan,cancela
0,6.0,15.0,11.0,1.7,3.4,3.0,2.0,1.0,70.58,Premium,0
1,46.0,7.0,10.0,2.2,2.6,1.0,10.0,2.0,115.68,Estandar,0
2,39.0,6.0,13.0,2.6,2.8,4.0,14.0,3.0,90.19,Basico,1
3,26.0,9.0,14.0,3.4,2.7,3.0,12.0,0.0,135.28,Estandar,0
4,26.0,17.0,10.0,4.1,4.0,2.0,13.0,3.0,66.31,Basico,1


In [4]:
# Limpieza.
df_dirty = df_raw.copy()

# Nulos aleatorios en campos que el usuario podría dejar vacíos en el formulario.
for col in ["horas_uso_diario", "calificacion_promedio", "precio_plan_mensual", "meses_suscrito"]:
    idx = rng.choice(df_dirty.index, size=int(N * 0.06), replace=False)
    df_dirty.loc[idx, col] = np.nan

# Calificaciones fuera del rango válido [1, 5].
idx_bajo = rng.choice(df_dirty.index, size=25, replace=False)
idx_alto = rng.choice(df_dirty.index, size=20, replace=False)
df_dirty.loc[idx_bajo, "calificacion_promedio"] = rng.uniform(-1, 0.9, size=25).round(1)
df_dirty.loc[idx_alto, "calificacion_promedio"] = rng.uniform(5.1, 8,  size=20).round(1)

# Precios negativos: dato imposible en el dominio de facturación.
idx_neg = rng.choice(df_dirty.index, size=15, replace=False)
df_dirty.loc[idx_neg, "precio_plan_mensual"] = rng.uniform(-50, 0, size=15).round(2)

# Duplicados por sincronización errónea entre sistemas.
duplicados = df_dirty.sample(n=25, random_state=SEED)
df_dirty   = pd.concat([df_dirty, duplicados], ignore_index=True)

print(f"Dataset con problemas: {df_dirty.shape}")
print(f"Nulos totales        : {df_dirty.isnull().sum().sum()}")
print(f"Duplicados           : {df_dirty.duplicated().sum()}")

Dataset con problemas: (1525, 11)
Nulos totales        : 364
Duplicados           : 25


---
## 2. Limpieza: Diagnóstico

Antes de corregir se cuantifica el estado actual del dataset.

In [5]:
print("Nulos por columna:")
print(df_dirty.isnull().sum())
print(f"\nDuplicados  : {df_dirty.duplicated().sum()}")
print(f"Total filas : {len(df_dirty)}")

Nulos por columna:
meses_suscrito             92
peliculas_vistas_mes        0
series_vistas_mes           0
horas_uso_diario           91
calificacion_promedio      91
dispositivos_vinculados     0
interrupciones_mes          0
quejas_soporte              0
precio_plan_mensual        90
tipo_plan                   0
cancela                     0
dtype: int64

Duplicados  : 25
Total filas : 1525


---
## 3. Limpieza: Estrategias de Imputación

A diferencia del cuaderno anterior (que usó `dropna`), aquí se **imputan**
los valores faltantes para no perder registros. La estrategia depende de
la distribución de cada columna:

| Distribución | Estrategia recomendada |
|---|---|
| Simétrica (normal) | Media |
| Asimétrica / con outliers | Mediana |
| Categórica | Moda |
| Valor de negocio claro | Constante |

In [6]:
df = df_dirty.copy()

# Imputación con MEDIANA: robusta ante precios extremos fuera de lo habitual.
for col in ["precio_plan_mensual", "horas_uso_diario"]:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f"  {col:<30} → mediana = {mediana:.2f}")

# Imputación con MEDIA: adecuada para calificaciones con distribución simétrica.
media_cal = df["calificacion_promedio"].mean()
df["calificacion_promedio"] = df["calificacion_promedio"].fillna(media_cal)
print(f"  {'calificacion_promedio':<30} → media   = {media_cal:.2f}")

# Imputación con CONSTANTE: suscriptor sin meses registrados = primer mes activo.
df["meses_suscrito"] = df["meses_suscrito"].fillna(1)
print(f"  {'meses_suscrito':<30} → constante = 1")

print(f"\nNulos restantes: {df.isnull().sum().sum()}")

  precio_plan_mensual            → mediana = 118.44
  horas_uso_diario               → mediana = 2.55
  calificacion_promedio          → media   = 3.47
  meses_suscrito                 → constante = 1

Nulos restantes: 0


In [7]:
# Imputación con MODA para la columna categórica.
# La moda es la categoría más frecuente en el dataset.
moda_plan = df["tipo_plan"].mode()[0]
df["tipo_plan"] = df["tipo_plan"].fillna(moda_plan)
print(f"Moda de tipo_plan: {moda_plan}")

Moda de tipo_plan: Basico


---
## 4. Limpieza: Reglas de Dominio

Valores que son sintácticamente válidos pero semánticamente imposibles.
`isnull()` no los detecta; requieren validación explícita por dominio.

In [8]:
# `clip` restringe la calificación al rango válido [1.0, 5.0].
# Una calificación de 0 o 6 no tiene sentido en una escala de estrellas 1-5.
df["calificacion_promedio"] = df["calificacion_promedio"].clip(lower=1.0, upper=5.0)
print("Calificaciones recortadas al rango [1.0, 5.0].")

Calificaciones recortadas al rango [1.0, 5.0].


In [9]:
# Precios negativos: se convierten a NaN y se reimputancon la mediana.
mask_neg = df["precio_plan_mensual"] < 0
print(f"Precios negativos encontrados: {mask_neg.sum()}")
df.loc[mask_neg, "precio_plan_mensual"] = np.nan
df["precio_plan_mensual"] = df["precio_plan_mensual"].fillna(df["precio_plan_mensual"].median())

Precios negativos encontrados: 18


In [10]:
# Eliminación de duplicados conservando la primera ocurrencia.
n_antes = len(df)
df      = df.drop_duplicates()
print(f"Duplicados eliminados: {n_antes - len(df)} | Filas finales: {len(df)}")

Duplicados eliminados: 25 | Filas finales: 1500


In [11]:
# Verificación final mediante aserciones
assert df.isnull().sum().sum() == 0,                                "Aún existen nulos."
assert (df["calificacion_promedio"].between(1.0, 5.0)).all(),       "Calificaciones fuera de rango."
assert (df["precio_plan_mensual"] >= 0).all(),                      "Precios negativos presentes."
print("Verificaciones superadas. Dataset listo para modelado.")
print(df[["precio_plan_mensual", "calificacion_promedio", "horas_uso_diario"]].describe())

Verificaciones superadas. Dataset listo para modelado.
       precio_plan_mensual  calificacion_promedio  horas_uso_diario
count          1500.000000            1500.000000       1500.000000
mean            119.592660               3.429282          2.547467
std              37.485762               0.958611          1.466089
min               5.990000               1.000000         -2.100000
25%              96.687500               2.800000          1.600000
50%             118.440000               3.470711          2.550000
75%             143.700000               4.100000          3.500000
max             281.030000               5.000000          7.100000


---
## 5. Codificación One-Hot (pd.get_dummies)

**One-Hot Encoding** crea una columna binaria (0/1) por cada categoría.
Es preferible a LabelEncoder cuando las categorías **no tienen orden natural**
(Basico, Estandar y Premium podrían tener orden, pero como el precio ya
está en el dataset no es necesario codificar ese orden aquí).

```
tipo_plan    →   tipo_plan_Estandar   tipo_plan_Premium
"Basico"     →        0                    0
"Estandar"   →        1                    0
"Premium"    →        0                    1
```
`drop_first=True` elimina la categoría de referencia (Basico) para evitar
multicolinealidad: si Estandar=0 y Premium=0, el plan es implícitamente Basico.

In [12]:
print("Columnas antes de OHE :", df.shape[1])
df = pd.get_dummies(df, columns=["tipo_plan"], drop_first=True, dtype=int)
print("Columnas después de OHE:", df.shape[1])

nuevas = [c for c in df.columns if "tipo_plan_" in c]
print("Columnas OHE generadas :", nuevas)
df[nuevas].value_counts().head()

Columnas antes de OHE : 11
Columnas después de OHE: 12
Columnas OHE generadas : ['tipo_plan_Estandar', 'tipo_plan_Premium']


,,count
tipo_plan_Estandar,tipo_plan_Premium,
0,0,514
1,0,497
0,1,489


In [13]:
# Tabla comparativa: LabelEncoder asigna enteros (introduce orden implícito).
# OHE crea columnas binarias independientes (sin orden entre categorías).
comparacion = pd.DataFrame({
    "tipo_plan"         : ["Basico", "Estandar", "Premium"],
    "LabelEncoder"      : [0, 1, 2],
    "OHE_Estandar"      : [0, 1, 0],
    "OHE_Premium"       : [0, 0, 1],
})
print(comparacion.to_string(index=False))
# Con LabelEncoder el modelo podría asumir que Premium (2) es "el doble"
# de Estandar (1) en lugar de tratarlos como categorías independientes.

tipo_plan  LabelEncoder  OHE_Estandar  OHE_Premium
   Basico             0             0            0
 Estandar             1             1            0
  Premium             2             0            1


---
## 6. División y Escalado

Mismos pasos que en el cuaderno anterior, incluidos aquí para que este
cuaderno sea autocontenido y funcione como referencia independiente.

In [14]:
FEATURES = [c for c in df.columns if c != "cancela"]
TARGET   = "cancela"

X = df[FEATURES].values
y = df[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Features ({len(FEATURES)}): {FEATURES}")

Train : (1200, 11)
Test  : (300, 11)
Features (11): ['meses_suscrito', 'peliculas_vistas_mes', 'series_vistas_mes', 'horas_uso_diario', 'calificacion_promedio', 'dispositivos_vinculados', 'interrupciones_mes', 'quejas_soporte', 'precio_plan_mensual', 'tipo_plan_Estandar', 'tipo_plan_Premium']


---
## 7. Entrenamiento del Modelo


In [15]:
modelo = RandomForestClassifier(
    n_estimators  = 100,  # → "Cantidad de árboles de decisión" en la UI
    max_depth     = 8,    # → "Profundidad máxima de los árboles" en la UI
    max_leaf_nodes= 40,   # → "Número máximo de hojas por árbol" en la UI
    random_state  = SEED,
    n_jobs        = -1,
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

print("Métricas con configuración inicial:")
print(f"  Exactitud : {accuracy_score (y_test, y_pred):.4f}")
print(f"  Precisión : {precision_score(y_test, y_pred):.4f}")
print(f"  Recall    : {recall_score   (y_test, y_pred):.4f}")
print(f"  F1 Score  : {f1_score       (y_test, y_pred):.4f}")

Métricas con configuración inicial:
  Exactitud : 0.8467
  Precisión : 0.7579
  Recall    : 0.7579
  F1 Score  : 0.7579


---
## 8. Persistencia del Modelo con joblib

El modelo se entrena en la pantalla "Entrenamiento" y se usa en la pantalla
"Predicciones". Como son dos contextos de ejecución distintos el modelo
debe **guardarse en disco** tras entrenar y **cargarse** antes de predecir.

Se guardan tres archivos:
- El modelo (`RandomForestClassifier` ajustado)
- El scaler (`StandardScaler` ajustado solo con datos de entrenamiento)
- La lista de features en el mismo orden exacto del entrenamiento

In [16]:
# El scaler es tan importante como el modelo: sin él, los datos nuevos
# llegarían en escala cruda al modelo que fue entrenado con datos escalados.
joblib.dump(modelo,   "modelo_streaming.joblib")
joblib.dump(scaler,   "scaler_streaming.joblib")
joblib.dump(FEATURES, "features_streaming.joblib")
print("Archivos guardados:")
print("  modelo_streaming.joblib")
print("  scaler_streaming.joblib")
print("  features_streaming.joblib")

Archivos guardados:
  modelo_streaming.joblib
  scaler_streaming.joblib
  features_streaming.joblib


In [17]:
# Carga desde disco:
# sin necesidad de reentrenar el modelo.
modelo_cargado    = joblib.load("modelo_streaming.joblib")
scaler_cargado    = joblib.load("scaler_streaming.joblib")
features_cargadas = joblib.load("features_streaming.joblib")

# El modelo cargado debe producir predicciones idénticas al original.
y_pred_cargado = modelo_cargado.predict(X_test)
assert np.array_equal(y_pred, y_pred_cargado), "Las predicciones difieren tras la carga."
print("Modelo cargado correctamente. Predicciones verificadas.")

Modelo cargado correctamente. Predicciones verificadas.


---
## 9. Predicción Individual con el Modelo Cargado

El usuario ingresa los campos del formulario; el backend construye el array,
aplica el scaler cargado y llama a `predict`.


In [18]:
# Perfil de un suscriptor con señales claras de posible cancelación.
datos_formulario = {
    "meses_suscrito"         : 3.0,
    "peliculas_vistas_mes"   : 1.0,    # muy poca actividad → señal de cancelación
    "series_vistas_mes"      : 0.0,
    "horas_uso_diario"       : 0.4,    # uso mínimo → señal de cancelación
    "calificacion_promedio"  : 2.0,    # valoración baja → señal de cancelación
    "dispositivos_vinculados": 1.0,
    "interrupciones_mes"     : 12.0,
    "quejas_soporte"         : 4.0,    # múltiples quejas → señal de cancelación
    "precio_plan_mensual"    : 160.0,
    # OHE de tipo_plan: "Premium" → Estandar=0, Premium=1
    "tipo_plan_Estandar"     : 0,
    "tipo_plan_Premium"      : 1,
}

In [19]:
# Construcción del array en el MISMO orden de columnas que el entrenamiento.
# Un orden incorrecto produce predicciones erróneas sin lanzar ningún error visible.
X_nuevo        = np.array([[datos_formulario[f] for f in features_cargadas]])
X_nuevo_scaled = scaler_cargado.transform(X_nuevo)

clase    = modelo_cargado.predict(X_nuevo_scaled)[0]
proba    = modelo_cargado.predict_proba(X_nuevo_scaled)[0]
etiqueta = "CANCELACIÓN PROBABLE" if clase == 1 else "CONTINUIDAD PROBABLE"

print("=" * 48)
print("   Resultado de la Predicción Individual")
print("=" * 48)
print(f"  Clase predicha   : {clase} → {etiqueta}")
print(f"  P(Continuidad)   : {proba[0]:.2%}")
print(f"  P(Cancelación)   : {proba[1]:.2%}")

   Resultado de la Predicción Individual
  Clase predicha   : 1 → CANCELACIÓN PROBABLE
  P(Continuidad)   : 6.11%
  P(Cancelación)   : 93.89%


---
## Resumen de Técnicas Cubiertas

| Técnica | Cuándo usarla |
|---|---|
| `fillna(mediana)` | Columnas numéricas con outliers o distribución asimétrica |
| `fillna(media)` | Columnas numéricas con distribución simétrica |
| `fillna(moda)` | Columnas categóricas |
| `fillna(constante)` | Cuando el nulo tiene un significado claro de negocio |
| `clip(lower, upper)` | Valores fuera de un rango físicamente posible |
| `pd.get_dummies` | Variables categóricas sin orden natural entre categorías |
| `joblib.dump / load` | Persistir el modelo |
